# Sequential Thesis Pipeline — 01 July 2026

This notebook orchestrates the three-stage ZA-GAS modelling pipeline described in the thesis.

---

## Pipeline Architecture

### Stage 1 — Standard GAS, no covariates
Twelve models: `{phi-only, phi+xi}` × `{short lags [1,2,3], seasonal lags [1,2,3,364–367]}` × `{unit, diagonal_inverse_fisher, inverse_fisher}` scaling.

**Objective:** select best GAS specification (scaling mode, lag set, TV parameters).

### Stage 2 — Standard GAS with weather covariates
Sequential, warm-started from Stage-1 winner:
1. Dew-point short (lags 1–3)
2. Dew-point seasonal (lags 1–3 + 364–367)
3. Dew-point + temperature short
4. Dew-point + temperature seasonal

**Objective:** select best weather covariate block.

### Stage 3 — Harvey Long-Short
Short component: Stage-2 winner weather block (X_short).  
Long component: 4 ENSO variants (none, 90-day mean, 90+30-day means, 90+30+daily lag).

**Objective:** select best Harvey specification.

---

## Execution Notes

- Models are checkpointed immediately after completion; resuming from interruption is automatic.
- All artifacts are saved under `artifacts/<run_id>/`.
- Running `run_pipeline(..., force_rerun=False)` (the default) skips completed models.
- To re-estimate from scratch, set `FORCE_RERUN = True`.

**Do not run until after code review and explicit authorization.**

## 0. Configuration

In [ ]:
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
ROOT          = Path(".")                        # repository root
ARTIFACTS_DIR = ROOT / "artifacts"               # output root
RUN_ID        = "run_20260701_bh"                # experiment identifier

# Precipitation data lives one level above FurtherTopics (confirmed from analysis.ipynb)
PRECIP_DIR    = Path(r"C:\Users\ilang\OneDrive\Documentos\Ilan\academia\dissertação\data\output")
ERA5_DIR      = ROOT / "data" / "input" / "ERA5"
NINO34_PATH   = ROOT / "data" / "processed" / "pacific" / "NINO34_daily.csv"

# ── Execution flags ──────────────────────────────────────────────────────────
FORCE_RERUN = False   # True: re-estimate everything; False: reuse cached artifacts
DO_STAGE1   = True
DO_STAGE2   = True
DO_STAGE3   = True

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Run ID   : {RUN_ID}")
print(f"Artifacts: {ARTIFACTS_DIR / RUN_ID}")
print(f"Force re-run: {FORCE_RERUN}")
print(f"Precip dir exists: {PRECIP_DIR.exists()}")
print(f"ERA5 dir exists  : {ERA5_DIR.exists()}")
print(f"NINO34 exists    : {NINO34_PATH.exists()}")

## 1. Imports

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "src"))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Framework imports
from data.loader import BHDataLoader
from distributions.gb2_log_link import GB2LogLink
from distributions.gb2_phi_only import GB2LogLinkPhiOnly
from pi_dynamics.factory import make_pi_dynamics
from pipeline.runner import run_pipeline, estimate_resources, get_logger

warnings.filterwarnings("ignore", category=RuntimeWarning)
print("Imports OK")

## 2. Load data

In [ ]:
loader = BHDataLoader(
    precip_dir  = PRECIP_DIR,
    era5_dir    = ERA5_DIR,
    nino34_path = NINO34_PATH,
)
data = loader.load_all()

print("=" * 50)
print("DATA SUMMARY")
print("=" * 50)
for k, v in data["summary"].items():
    print(f"  {k:<25}: {v}")

print("\nCovariate blocks available:")
for name, arr in data["covariate_blocks_train"].items():
    print(f"  {name:<30}: shape {arr.shape}")

## 3. Build model components

In [ ]:
# Phi-only: only scale parameter phi is time-varying; xi is static.
# GASFilter reads default_static_params from GB2LogLinkPhiOnly → ["xi","gamma","zeta"].
dist_phi   = GB2LogLinkPhiOnly()

# Phi+xi: both scale (phi) and shape (xi) are time-varying; gamma and zeta static.
dist_phixi = GB2LogLink()

# Occurrence model: AR-logistic pi dynamics on SEASONAL lags [1, 365, 366].
# Lag set for pi is fixed per constants.SEASONAL_LAGS["daily"]; not varied in Stage 1.
pi_dyn = make_pi_dynamics("ar_logistic", seasonal="daily")

print(f"dist_phi   : {dist_phi.__class__.__name__}  TV={dist_phi.tv_param_names}  static={dist_phi.default_static_params}")
print(f"dist_phixi : {dist_phixi.__class__.__name__}  TV={dist_phixi.tv_param_names}")
print(f"pi_dyn     : {pi_dyn.__class__.__name__}  params={pi_dyn.param_names('daily')}")

## 4. Resource estimate

Before running, display an estimated wall-time and memory cost for each stage.
These estimates assume ~5 minutes per Stage-1 model on an 8-core machine with
the BFGS optimizer and a typical 3 650-day training series.

In [ ]:
from pipeline.runner import estimate_resources

print("Stage 1  (12 models, no covariates)")
print(estimate_resources(n_models=12, avg_runtime_s=300, mb_per_model=200))

print("\nStage 2  (4 models, weather covariates, sequential)")
print(estimate_resources(n_models=4,  avg_runtime_s=360, mb_per_model=200))

print("\nStage 3  (4 models, Harvey long-short)")
print(estimate_resources(n_models=4,  avg_runtime_s=480, mb_per_model=300))

total_min = (12*300 + 4*360 + 4*480) / 60
print(f"\nTotal sequential estimate: {total_min:.0f} min  ({total_min/60:.1f} h)")

## 5. Run the pipeline

> **Authorization required.** Do not execute this cell until you have reviewed
> `docs/AUDIT_01072026.md` and explicitly authorized running.
>
> Running with `FORCE_RERUN=False` (default) is idempotent:
> completed models are skipped and loaded from cache.
> You can interrupt and resume at any time.

Expected outputs:
- `artifacts/run_20260701_bh/stage{1,2,3}/<model_id>/metadata.json`
- `artifacts/run_20260701_bh/stage_winners.json`
- `artifacts/run_20260701_bh/execution_log.jsonl`

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

pipeline_output = run_pipeline(
    data          = data,
    pi_dyn        = pi_dyn,
    dist_phi      = dist_phi,
    dist_phixi    = dist_phixi,
    artifacts_dir = ARTIFACTS_DIR,
    run_id        = RUN_ID,
    force_rerun   = FORCE_RERUN,
    do_stage1     = DO_STAGE1,
    do_stage2     = DO_STAGE2,
    do_stage3     = DO_STAGE3,
)

print("\nPipeline complete.")
print(f"Run directory: {pipeline_output['run_dir']}")

## 6. Results Summary

In [ ]:
def _fmt(r):
    if r is None:
        return "(none)"
    ll   = r.get("loglik",  float("nan"))
    crps = r.get("oos_metrics", {}).get("crps_mean", float("nan"))
    return f"{r['model_id']}  |  loglik={ll:.2f}  |  crps={crps:.4f}  |  validity={r.get('validity')}"

print("Stage 1 winner:", _fmt(pipeline_output["stage1_winner"]))
print("Stage 2 winner:", _fmt(pipeline_output["stage2_winner"]))
print("Stage 3 winner:", _fmt(pipeline_output["stage3_winner"]))

In [ ]:
# Stage 1 full ranking
def _to_row(r):
    oos = r.get("oos_metrics", {})
    return {
        "model_id": r.get("model_id"),
        "validity": r.get("validity"),
        "loglik":   round(r.get("loglik", float("nan")), 2),
        "crps":     round(oos.get("crps_mean", float("nan")), 5),
        "rmse":     round(oos.get("rmse",      float("nan")), 4),
    }

stage1_df = pd.DataFrame([_to_row(r) for r in pipeline_output["stage1_results"]])
stage1_df = stage1_df.sort_values("loglik", ascending=False)
print("\n=== STAGE 1 RANKING ===")
display(stage1_df)

In [ ]:
# Stage 2 ranking
stage2_df = pd.DataFrame([_to_row(r) for r in pipeline_output["stage2_results"]])
stage2_df = stage2_df.sort_values("loglik", ascending=False)
print("=== STAGE 2 RANKING ===")
display(stage2_df)

In [ ]:
# Stage 3 ranking
stage3_df = pd.DataFrame([_to_row(r) for r in pipeline_output["stage3_results"]])
stage3_df = stage3_df.sort_values("loglik", ascending=False)
print("=== STAGE 3 RANKING ===")
display(stage3_df)

## 7. Stage 1 — Detailed results

Log-likelihood across 12 configurations, grouped by scaling mode.

In [ ]:
if pipeline_output["stage1_results"]:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

    for ax, tv in zip(axes, ["phi", "phixi"]):
        rows = [r for r in pipeline_output["stage1_results"]
                if f"_{tv}_" in r.get("model_id", "")]
        labels = [r["model_id"].replace(f"stage1_{tv}_", "") for r in rows]
        lls    = [r.get("loglik", float("nan")) for r in rows]
        colors = ["steelblue" if "valid_converged" in str(r.get("validity")) else
                  "orange"    if "warning"        in str(r.get("validity")) else
                  "red"       for r in rows]
        ax.bar(labels, lls, color=colors)
        ax.set_title(f"TV params: {tv}")
        ax.set_xlabel("lag_scaling")
        ax.set_ylabel("log-likelihood")
        ax.tick_params(axis="x", rotation=30)

    fig.suptitle("Stage 1 — log-likelihood by configuration", fontsize=13)
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / RUN_ID / "stage1_loglik_comparison.pdf", bbox_inches="tight")
    plt.show()
    print("Figure saved.")

## 8. Stage 2 — Weather covariate effect

Log-likelihood improvement over Stage-1 winner as covariates are added sequentially.

In [ ]:
if pipeline_output["stage1_winner"] and pipeline_output["stage2_results"]:
    ll_base = pipeline_output["stage1_winner"].get("loglik", float("nan"))
    labels  = [r["model_id"].replace("stage2_", "") for r in pipeline_output["stage2_results"]]
    delta   = [r.get("loglik", float("nan")) - ll_base for r in pipeline_output["stage2_results"]]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(labels, delta, color="teal")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xlabel("Weather covariate block")
    ax.set_ylabel("ΔlogL vs Stage-1 winner")
    ax.set_title("Stage 2 — Gain from weather covariates")
    ax.tick_params(axis="x", rotation=20)
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / RUN_ID / "stage2_delta_loglik.pdf", bbox_inches="tight")
    plt.show()

## 9. Stage 3 — Harvey long-short + ENSO

Log-likelihood gain from Harvey decomposition vs Stage-2 winner.

In [ ]:
if pipeline_output["stage2_winner"] and pipeline_output["stage3_results"]:
    ll_base = pipeline_output["stage2_winner"].get("loglik", float("nan"))
    labels  = [r["model_id"].replace("stage3_harvey_", "") for r in pipeline_output["stage3_results"]]
    delta   = [r.get("loglik", float("nan")) - ll_base for r in pipeline_output["stage3_results"]]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(labels, delta, color="darkslategray")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xlabel("Harvey ENSO variant")
    ax.set_ylabel("ΔlogL vs Stage-2 winner")
    ax.set_title("Stage 3 — Harvey + ENSO gain")
    ax.tick_params(axis="x", rotation=15)
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / RUN_ID / "stage3_delta_loglik.pdf", bbox_inches="tight")
    plt.show()

## 10. OOS CRPS Comparison

Mean CRPS (lower = better) across all estimated models.

In [ ]:
all_results = (
    pipeline_output["stage1_results"]
    + pipeline_output["stage2_results"]
    + pipeline_output["stage3_results"]
)

crps_rows = [
    {"model_id": r["model_id"],
     "stage": r["model_id"].split("_")[0] + "_" + r["model_id"].split("_")[1],
     "crps":  r.get("oos_metrics", {}).get("crps_mean", float("nan")),
     "validity": r.get("validity"),
    }
    for r in all_results if r.get("status") != "failed"
]
crps_df = pd.DataFrame(crps_rows).sort_values("crps")
print("=== OOS CRPS RANKING (lower is better) ===")
display(crps_df.head(10))

## 11. Persistence and Score Loading — Stage 3 Winner

Extract and display the Harvey model parameters from the Stage-3 winner.

In [ ]:
w3 = pipeline_output.get("stage3_winner")
if w3 and w3.get("result"):
    names  = w3["result"].get("param_names", [])
    theta  = w3["result"]["theta"]
    se     = w3["result"].get("std_errors")
    
    param_df = pd.DataFrame({"parameter": names, "estimate": theta})
    if se is not None:
        param_df["std_error"] = se
        param_df["t_stat"]    = param_df["estimate"] / param_df["std_error"]
    
    # Filter to Harvey-specific parameters (Harvey recursion)
    harvey_mask = param_df["parameter"].str.match(r"(A_|B_|omega_|f0_|L0_|S0_|gamma_)")
    display(param_df[harvey_mask].set_index("parameter").round(4))
elif w3:
    # Cached: load from artifact
    run_path = Path(pipeline_output["run_dir"]) / "stage3" / w3["model_id"] / "estimated_parameters.csv"
    if run_path.exists():
        display(pd.read_csv(run_path).set_index("parameter").round(4))
    else:
        print("Artifact not found:", run_path)
else:
    print("Stage 3 not yet completed.")

## 12. Execution log inspection

In [ ]:
log_path = Path(pipeline_output["run_dir"]) / "execution_log.jsonl"
if log_path.exists():
    events = [json.loads(line) for line in log_path.read_text().splitlines() if line.strip()]
    log_df = pd.DataFrame(events)
    print(f"Total log events: {len(log_df)}")
    done = log_df[log_df["event"] == "done"] if "event" in log_df.columns else log_df
    if len(done):
        total_s = done["runtime_s"].sum() if "runtime_s" in done.columns else 0
        print(f"Total fit time: {total_s/60:.1f} min")
        display(done[["model_id", "validity", "loglik", "crps_mean", "runtime_s"]]
               .sort_values("loglik", ascending=False)
               .round(3))
else:
    print("Execution log not found (pipeline not yet run?).")

## 13. Stage winners JSON

Full summary table saved to `stage_winners.json`.

In [ ]:
winners_path = Path(pipeline_output["run_dir"]) / "stage_winners.json"
if winners_path.exists():
    winners = json.loads(winners_path.read_text())
    for stage, info in winners.items():
        print(f"\n{stage.upper()} WINNER: {info.get('winner')}")
        print(f"  loglik = {info.get('loglik')}  crps = {info.get('crps')}")
else:
    print("stage_winners.json not found (pipeline not yet run?).")